In [1]:
import re, unicodedata
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
from IPython.display import display

# Paths
DATA_DIR = Path("../Normalized_Data")
SRC = DATA_DIR / "sinhala-hate-speech-dataset_normalized.csv"
DATA_DIR2 = Path("../Pre processed Data")
DST_CLEAN = DATA_DIR2 /"sinhala-hate-speech-Clean.csv"


# Options
DROP_EXACT_DUPLICATES = True # drop exact duplicate rows
TEST_SIZE = 0.2 # 20% of the data will be kept for evaluation, and 80% will be used for training.
RANDOM_STATE = 42 # random seed for reproducibility
URL_RX = re.compile(r"(?xi)\b((?:https?://|www\d{0,3}[.])[^\s<>\"']+)") # regex to detect URLs
EMAIL_RX = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.I) # regex to detect email addresses
MENTION_RX = re.compile(r"(^|\s)@\w+") # regex to detect Twitter mentions
HASHTAG_RX = re.compile(r"(^|\s)#\w+") # regex to detect hashtags
MULTI_WS_RX = re.compile(r"\s+") # regex to detect multiple whitespace characters



# shrink 3 or more consecutive identical letters to 2 letters 
def shrink_repeats_en(t: str) -> str:
    return re.sub(r"([A-Za-z])\1{2,}", lambda m: m.group(1) * 2, t) 

def clean_row(text: str, lang: str) -> str:
    # Main Normalization forms are NFC, NFD, NFKC, and NFKD.
    # NFC: Normalization Form C (Canonical Composition)
    # NFD: Normalization Form D (Canonical Decomposition)
    # NFKC: Normalization Form KC (Compatibility Composition)
    # NFKD: Normalization Form KD (Compatibility Decomposition)
    # We use NFC here to ensure that characters are in their composed form.
    t = unicodedata.normalize("NFC", str(text)) # Standardize text representation using Unicode Normalization Form C, which composes characters to their canonical form.
    t = URL_RX.sub(" ", t)
    t = EMAIL_RX.sub(" ", t)
    t = MENTION_RX.sub(" ", t)
    t = HASHTAG_RX.sub(" ", t)
    if str(lang).strip().lower() == "en":
        t = t.lower()
        t = shrink_repeats_en(t)
    t = MULTI_WS_RX.sub(" ", t).strip() # Collapse multiple whitespace characters into a single space
    return t

# read the "merged_all.csv" file
assert SRC.exists(), f"Input not found: {SRC}"
df = pd.read_csv(SRC)

# verify required columns
assert {"text","label","lang"}.issubset(df.columns), df.columns
print(f"Loaded {len(df)} rows from {SRC}")
display(df.head())


# Remove exact duplicate rows
if DROP_EXACT_DUPLICATES:
    before = len(df)
    df = df.drop_duplicates(subset=["text","label","lang"]).reset_index(drop=True)
    after = len(df)
    print(f"Duplicated {before - after} rows (now {after}).")


# Clean the text data by row by row
df["text"] = df.apply(lambda r: clean_row(r["text"], r["lang"]), axis=1)
# Remove empty text rows
df = df[df["text"].astype(str).str.len() > 0].reset_index(drop=True)
print(f"After cleaning: {len(df)} rows")
display(df.head(10))

# save the cleaned dataset as "merged_clean.csv"
# Ensure output directory exists
DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(DST_CLEAN, index=False)
print(f"Saved cleaned dataset to: {DST_CLEAN}")


# display the data counts by label and language
print("\nLabel distribution (0=neutral, 1=hate):")
print(df["label"].value_counts(dropna=False))

print("\nBy language:")
print(df["lang"].value_counts(dropna=False))

print("\nCross tab (lang x label):")
print(pd.crosstab(df["lang"], df["label"]))


Loaded 6309 rows from ..\Normalized_Data\sinhala-hate-speech-dataset_normalized.csv


,text,label,lang
0,`හුදෙකලා´ එකේ `ද´ ඇහෙනවද කාටහරි,0,si
1,18ක පෙනුමක් තියෙන්නේ,0,si
2,2020 එනවා කිව්නාට මෙහෙම එයි කියලා හිතුවේ නෑ පට්ට,0,si
3,25 කැල්ල..හම්මෝ..,0,si
4,25 කෙල්ල අයියලගෙ කාලද කොහෙද ඉන්නෙ,1,si


Duplicated 0 rows (now 6309).
After cleaning: 6309 rows


,text,label,lang
0,`හුදෙකලා´ එකේ `ද´ ඇහෙනවද කාටහරි,0,si
1,18ක පෙනුමක් තියෙන්නේ,0,si
2,2020 එනවා කිව්නාට මෙහෙම එයි කියලා හිතුවේ නෑ පට්ට,0,si
3,25 කැල්ල..හම්මෝ..,0,si
4,25 කෙල්ල අයියලගෙ කාලද කොහෙද ඉන්නෙ,1,si
5,25 ක් වගේ ???ඉතින් වැලලියන්,1,si
6,25 ලුක් එක දැකල ඔලුව සවුත්තු උන තරිදු...,0,si
7,25ක පෙනුමක් කියපු කෑල්ලේ ඉදන් බලන එක නතර කලා,0,si
8,33ක් ගුවන්තොටුපළවල ලෝකයේ හිරවෙලා ඉන්නවා ඩුබායි...,0,si
9,5 දෙනා කිව්වෙ මේ පස්දෙනාද..,0,si


Saved cleaned dataset to: ..\Pre processed Data\sinhala-hate-speech-Clean.csv

Label distribution (0=neutral, 1=hate):
label
1    3446
0    2863
Name: count, dtype: int64

By language:
lang
si    6309
Name: count, dtype: int64

Cross tab (lang x label):
label     0     1
lang             
si     2863  3446
